In [33]:
!pip install -U selenium pandas pyautogui


[notice] A new release of pip is available: 26.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [50]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
import pandas as pd
import time
import os
import sys
import traceback

options = webdriver.ChromeOptions()

# Use a dedicated Chrome profile to retain the login session.
# Do not use this profile in another Chrome instance simultaneously.
options.add_argument(
    "--user-data-dir=/Users/apple/whatsapp_selenium_profile"
)

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 30)

driver.get("https://web.whatsapp.com/")

input("Scan the WhatsApp QR code if required, then press Enter here...")

print("Python:", sys.version)
print("Selenium:", __import__("selenium").__version__)
print("Chrome:", driver.capabilities.get("browserVersion"))
print(
    "ChromeDriver:",
    driver.capabilities.get("chrome", {}).get("chromedriverVersion")
)

Python: 3.14.4 (main, Apr  7 2026, 13:13:20) [Clang 21.0.0 (clang-2100.0.123.102)]
Selenium: 4.49.0
Chrome: 155.0.8059.12
ChromeDriver: 155.0.8059.12 (384c82079f60e0b79cdca6e49efd5cfceefa5b74-refs/branch-heads/8059@{#450})


In [51]:
csv_path = (
    "/Users/apple/Documents/Personal Projects/"
    "WhatsApp_Bulk_Graphic_Sender/test_file.csv"
)

df = pd.read_csv(csv_path, dtype=str).fillna("")

required_columns = ["Name", "Mobile Number With Code"]

missing = [col for col in required_columns if col not in df.columns]

if missing:
    raise ValueError(f"Missing CSV columns: {missing}")

print(df[required_columns].head())
print("Total contacts:", len(df))

    Name Mobile Number With Code
0  test1           +916375685054
1  test2           +919461289390
2  test3           +917014257910
Total contacts: 3


In [52]:
# def send_message(name, number, message, image_path="", video_path=""):
#     from selenium.webdriver.common.by import By
#     from selenium.webdriver.support.ui import WebDriverWait
#     from selenium.webdriver.support import expected_conditions as EC
#     from selenium.common.exceptions import TimeoutException
#     import os
#     import time
#     import mimetypes

#     wait = WebDriverWait(driver, 30)

#     try:
#         # =========================================================
#         # VALIDATE
#         # =========================================================

#         phone = "".join(filter(str.isdigit, str(number)))

#         if image_path and video_path:
#             print("ERROR: image_path and video_path cannot both be used.")
#             return False

#         file_path = image_path if image_path else video_path

#         if not file_path:
#             print("ERROR: No image/video supplied.")
#             return False

#         file_path = os.path.abspath(file_path)

#         if not os.path.isfile(file_path):
#             print(f"ERROR: File does not exist: {file_path}")
#             return False

#         mime_type = mimetypes.guess_type(file_path)[0]

#         print("\n========== FILE ==========")
#         print(f"Path : {file_path}")
#         print(f"MIME : {mime_type}")
#         print(f"Size : {os.path.getsize(file_path)}")
#         print("==========================")

#         # =========================================================
#         # OPEN CHAT
#         # =========================================================

#         driver.get(
#             f"https://web.whatsapp.com/send?phone={phone}"
#         )

#         wait.until(
#             EC.presence_of_element_located(
#                 (
#                     By.XPATH,
#                     '//footer//div[@contenteditable="true"]'
#                 )
#             )
#         )

#         print(f"Chat opened for {name}")

#         # =========================================================
#         # IMPORTANT:
#         # DO NOT CLICK ATTACH
#         # DO NOT CLICK PHOTOS & VIDEOS
#         #
#         # We directly target WhatsApp's image file input.
#         # =========================================================

#         if image_path:

#             print("Searching directly for image input...")

#             image_input = None

#             # First try exact accept=image/*
#             try:
#                 image_input = wait.until(
#                     EC.presence_of_element_located(
#                         (
#                             By.XPATH,
#                             '//input[@type="file" and '
#                             '@accept="image/*"]'
#                         )
#                     )
#                 )

#                 print(
#                     "Found exact image input: accept=image/*"
#                 )

#             except TimeoutException:

#                 print(
#                     "Exact image input not immediately found."
#                 )

#                 # Fallback
#                 inputs = driver.find_elements(
#                     By.XPATH,
#                     '//input[@type="file"]'
#                 )

#                 print(
#                     f"File inputs found: {len(inputs)}"
#                 )

#                 for i, element in enumerate(inputs):

#                     accept = element.get_attribute("accept")

#                     print(
#                         f"Input {i}: accept={accept}"
#                     )

#                     if accept and accept.strip().lower() == "image/*":
#                         image_input = element
#                         print(
#                             f"Selected image input #{i}"
#                         )
#                         break

#             if image_input is None:

#                 print(
#                     "ERROR: WhatsApp image input not found."
#                 )

#                 driver.save_screenshot(
#                     "image_input_not_found.png"
#                 )

#                 return False

#             # =====================================================
#             # DIRECT FILE UPLOAD
#             # =====================================================

#             print(
#                 "Sending file directly to WhatsApp image input..."
#             )

#             image_input.send_keys(file_path)

#             print(
#                 "send_keys() completed."
#             )

#         # =========================================================
#         # VIDEO
#         # =========================================================

#         elif video_path:

#             print("Searching for video input...")

#             video_input = None

#             inputs = driver.find_elements(
#                 By.XPATH,
#                 '//input[@type="file"]'
#             )

#             for i, element in enumerate(inputs):

#                 accept = element.get_attribute("accept")

#                 print(
#                     f"Input {i}: accept={accept}"
#                 )

#                 if (
#                     accept
#                     and "video" in accept.lower()
#                 ):
#                     video_input = element
#                     print(
#                         f"Selected video input #{i}"
#                     )
#                     break

#             if video_input is None:

#                 print(
#                     "ERROR: Video input not found."
#                 )

#                 return False

#             video_input.send_keys(file_path)

#             print(
#                 "Video uploaded."
#             )

#         # =========================================================
#         # WAIT FOR MEDIA PREVIEW
#         # =========================================================

#         print(
#             "Waiting for WhatsApp media preview..."
#         )

#         time.sleep(3)

#         # =========================================================
#         # DEBUG CURRENT PAGE
#         # =========================================================

#         print(
#             "Checking for media Send button..."
#         )

#         send_xpath = (
#             '//div[@role="button" and '
#             'starts-with(@aria-label,"Send")]'
#         )

#         send_button = wait.until(
#             EC.element_to_be_clickable(
#                 (By.XPATH, send_xpath)
#             )
#         )

#         print(
#             "Media Send button found."
#         )

#         # =========================================================
#         # CAPTION
#         # =========================================================

#         if message and message.strip():

#             print("Adding caption...")

#             editables = []

#             for element in driver.find_elements(
#                 By.XPATH,
#                 '//div[@contenteditable="true"]'
#             ):

#                 try:
#                     if (
#                         element.is_displayed()
#                         and element.is_enabled()
#                     ):
#                         editables.append(element)

#                 except Exception:
#                     pass

#             if editables:

#                 caption = editables[-1]

#                 caption.click()
#                 caption.send_keys(message)

#                 print(
#                     "Caption entered."
#                 )

#         # =========================================================
#         # SEND
#         # =========================================================

#         time.sleep(1)

#         send_button = wait.until(
#             EC.element_to_be_clickable(
#                 (By.XPATH, send_xpath)
#             )
#         )

#         send_button.click()

#         print(
#             f"Photo/video sent to {name}"
#         )

#         return True

#     except TimeoutException:

#         print(
#             f"Timed out while sending to {name}"
#         )

#         driver.save_screenshot(
#             "whatsapp_timeout.png"
#         )

#         return False

#     except Exception as e:

#         print(
#             f"Error sending to {name}: {e}"
#         )

#         driver.save_screenshot(
#             "whatsapp_error.png"
#         )

#         return False

In [53]:
def send_message(name, number, message, image_path="", video_path=""):
    try:
        phone = str(number).strip().replace("+", "")

        driver.get(
            f"https://web.whatsapp.com/send?phone={phone}"
        )

        time.sleep(4)

        # Wait for chat
        wait.until(
            EC.presence_of_element_located(
                (By.XPATH, '//div[@contenteditable="true"]')
            )
        )

        # -------------------------------------------------
        # IMAGE UPLOAD
        # -------------------------------------------------
        if image_path:

            attach = wait.until(
                EC.element_to_be_clickable(
                    (By.XPATH, '//*[@aria-label="Attach"]')
                )
            )
            attach.click()

            time.sleep(1)

            photo_video = wait.until(
                EC.presence_of_element_located(
                    (
                        By.XPATH,
                        '//*[normalize-space(text())="Photos & videos"]'
                    )
                )
            )

            photo_video.click()

            time.sleep(1)

            # IMPORTANT:
            # Use the Photos & Videos combined input,
            # not the image-only input.
            inputs = driver.find_elements(
                By.XPATH,
                '//input[@type="file"]'
            )

            media_input = None

            for inp in inputs:
                accept = inp.get_attribute("accept") or ""

                if (
                    "image/*" in accept
                    and "video/mp4" in accept
                ):
                    media_input = inp
                    break

            if media_input is None:
                raise Exception(
                    "Photos & Videos file input not found."
                )

            media_input.send_keys(
                os.path.abspath(image_path)
            )

            # Wait for selected media preview
            send_button = wait.until(
                EC.presence_of_element_located(
                    (
                        By.XPATH,
                        '//div[@role="button" and starts-with(@aria-label,"Send")]'
                    )
                )
            )

            time.sleep(2)

            # Add caption if provided
            if message:

                caption_box = wait.until(
                    EC.presence_of_element_located(
                        (
                            By.XPATH,
                            '//div[@contenteditable="true"]'
                        )
                    )
                )

                caption_box.click()
                caption_box.send_keys(message)

            send_button.click()

            time.sleep(3)

            return True

        # -------------------------------------------------
        # TEXT ONLY
        # -------------------------------------------------
        message_box = wait.until(
            EC.presence_of_element_located(
                (
                    By.XPATH,
                    '//div[@contenteditable="true"]'
                )
            )
        )

        message_box.click()
        message_box.send_keys(message)

        message_box.send_keys(
            Keys.ENTER
        )

        time.sleep(2)

        return True

    except Exception as e:

        print(
            f"Failed to send message to {name}: {e}"
        )

        traceback.print_exc()

        return False

In [57]:
image_path = (
    "/Users/apple/Documents/Personal Projects/"
    "WhatsApp_Bulk_Graphic_Sender/invitation.jpeg"
)

video_path = ""

for index, row in df.iterrows():
    name = str(row["Name"]).strip()
    number = str(row["Mobile Number With Code"]).strip()

    message = ""

    print(f"\nProcessing {index + 1}: {name}")

    success = send_message(
        name=name,
        number=number,
        message=message,
        image_path=image_path,
        video_path=video_path
    )

    if success:
        print(f"Completed processing for {name}")
    else:
        print(f"Failed processing for {name}")

    # Avoid sending requests too rapidly
    time.sleep(5)


Processing 1: test1
Completed processing for test1

Processing 2: test2
Completed processing for test2

Processing 3: test3
Completed processing for test3


In [58]:
# Close browser
driver.quit()